# 🔍 MODEL 1: Phát hiện Bất thường / Suy giảm Tức thời bằng Extended Isolation Forest (EIF)

---
### 📌 Cơ chế Tự Chọn Đặc trưng từ Unified Dataset (`aiops_network_dataset.csv`):
- **File Dataset dùng chung cho cả 3 Model:** `aiops_network_dataset.csv` (`700 mẫu × 117 cột`).
- **Tập đặc trưng Model 1 tự chọn (`14 KPIs tức thời`):** Chọn 14 chỉ số đại diện sức khỏe tức thời của toàn mạng & dịch vụ tại thời điểm $t$ (`KEY_TIMESERIES_KPIS`).
- **Cột Nhãn Mục tiêu (`Target`) Model 1 tự chọn:** `current_degraded` (`0`: Hiện tại ổn định, `1`: Hiện tại đang suy giảm).
- **File Model xuất ra:** `model1_eif_detector.pkl`

In [ ]:
import glob
import os
import warnings
from typing import List, Optional

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import RobustScaler
from sklearn.svm import OneClassSVM

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

## 1. Cài đặt Thuật toán Extended Isolation Forest (EIF - Random Oblique Hyperplane Cuts)

In [ ]:
class ExtendedIsolationForest:
    """Extended Isolation Forest (EIF) sử dụng các siêu phẳng nghiêng ngẫu nhiên (Oblique Hyperplanes)."""
    def __init__(self, n_estimators: int = 360, n_rotations: int = 12, extension_level: Optional[int] = None, contamination: float = 0.06, random_state: int = 42):
        self.n_estimators = n_estimators
        self.n_rotations = n_rotations
        self.extension_level = extension_level
        self.contamination = contamination
        self.random_state = random_state
        self.projection_matrices_: List[np.ndarray] = []
        self.forests_: List[IsolationForest] = []
        self.threshold_: float = 0.0

    def _build_oblique_projection(self, n_features: int, rng: np.random.Generator) -> np.ndarray:
        ext = self.extension_level if self.extension_level is not None else max(1, n_features - 1)
        ext = min(ext, n_features - 1)
        proj = rng.standard_normal((n_features, n_features))
        if ext < n_features - 1:
            for col in range(n_features):
                zero_idx = rng.choice(n_features, size=(n_features - ext - 1), replace=False)
                proj[zero_idx, col] = 0.0
        norms = np.linalg.norm(proj, axis=0, keepdims=True)
        norms = np.where(norms == 0, 1.0, norms)
        return proj / norms

    def fit(self, X: np.ndarray) -> 'ExtendedIsolationForest':
        X = np.asarray(X, dtype=float)
        n_features = X.shape[1]
        rng = np.random.default_rng(self.random_state)
        self.projection_matrices_ = []
        self.forests_ = []
        trees_per_rot = max(25, self.n_estimators // self.n_rotations)
        for i in range(self.n_rotations):
            R = self._build_oblique_projection(n_features, rng)
            iso = IsolationForest(n_estimators=trees_per_rot, contamination=self.contamination, random_state=self.random_state + i, n_jobs=-1)
            iso.fit(X @ R)
            self.projection_matrices_.append(R)
            self.forests_.append(iso)
        scores = self.anomaly_score(X)
        self.threshold_ = float(np.percentile(scores, 100.0 * (1.0 - self.contamination)))
        return self

    def anomaly_score(self, X: np.ndarray) -> np.ndarray:
        X = np.asarray(X, dtype=float)
        total_scores = np.zeros(X.shape[0], dtype=float)
        for R, iso in zip(self.projection_matrices_, self.forests_):
            total_scores += -iso.score_samples(X @ R)
        return total_scores / max(1, len(self.forests_))

    def predict(self, X: np.ndarray) -> np.ndarray:
        return (self.anomaly_score(X) >= self.threshold_).astype(int)

## 2. Đọc Unified Dataset (`aiops_network_dataset.csv`) & Tự Chọn Đặc trưng cho Model 1

In [ ]:
candidates = (
    glob.glob('/kaggle/input/**/aiops_network_dataset.csv', recursive=True)
    + ['../../dataset/aiops_network_dataset.csv', '../dataset/aiops_network_dataset.csv', 'aiops_network_dataset.csv']
)
csv_path = next((p for p in candidates if os.path.exists(p)), None)
assert csv_path is not None, 'Hãy upload file aiops_network_dataset.csv lên Kaggle!'

df = pd.read_csv(csv_path).replace([np.inf, -np.inf], np.nan).fillna(0.0)

# Model 1 tự chọn 14 KPIs tức thời và cột nhãn current_degraded
MODEL1_FEATURES = [
    'wan1_rx_bytes_rate', 'wan1_tx_bytes_rate', 'branch_tcp_retrans_rate',
    'wan_tcp_syn_retrans_rate', 'wan_tcp_timeouts_rate', 'fw_rx_pkts_rate',
    'fw_avg_pkt_size', 'client_lan_tx_pkts_rate', 'srv_tcp_out_rsts_rate',
    'srv_tcp_passive_opens_rate', 'srv_cpu_usage', 'nginx_req_rate',
    'api_req_rate', 'api_latency_avg'
]
TARGET_COL = 'current_degraded'

print(f'✅ Đã nạp Unified Dataset: {csv_path} (Tổng {df.shape[0]} mẫu x {df.shape[1]} cột)')
print(f'🔍 Model 1 tự động chọn : {len(MODEL1_FEATURES)} đặc trưng KPIs tức thời | Target: {TARGET_COL}')
display(df[['datetime', *MODEL1_FEATURES[:6], TARGET_COL]].head())

## 3. Thực nghiệm Đối sánh: Extended Isolation Forest (EIF) vs Standard Isolation Forest vs OC-SVM vs LOF

In [ ]:
X = df[MODEL1_FEATURES].values
y = df[TARGET_COL].values.astype(int)

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

scaler = RobustScaler().fit(X_tr[y_tr == 0])
X_tr_s = np.arcsinh(scaler.transform(X_tr))
X_te_s = np.arcsinh(scaler.transform(X_te))
X_tr_normal = X_tr_s[y_tr == 0]

# 1. Extended Isolation Forest (EIF)
eif = ExtendedIsolationForest(n_estimators=360, n_rotations=12, extension_level=len(MODEL1_FEATURES)-1, contamination=0.06, random_state=42)
eif.fit(X_tr_normal)
tr_scores = eif.anomaly_score(X_tr_s)
best_th, best_f1 = eif.threshold_, -1.0
for q in np.linspace(30, 65, 71):
    th = float(np.percentile(tr_scores, q))
    f1_c = f1_score(y_tr, (tr_scores >= th).astype(int), zero_division=0)
    if f1_c > best_f1:
        best_f1, best_th = f1_c, th
eif.threshold_ = best_th
eif_preds = eif.predict(X_te_s)
eif_scores = eif.anomaly_score(X_te_s)

# 2. Standard Isolation Forest
std_iso = IsolationForest(n_estimators=300, contamination=0.05, random_state=42, n_jobs=-1).fit(X_tr_normal)
std_preds = np.where(std_iso.predict(X_te_s) == -1, 1, 0)
std_scores = -std_iso.score_samples(X_te_s)

# 3. One-Class SVM
ocsvm = OneClassSVM(kernel='rbf', nu=0.06, gamma='scale').fit(X_tr_normal)
svm_preds = np.where(ocsvm.predict(X_te_s) == -1, 1, 0)
svm_scores = -ocsvm.decision_function(X_te_s)

# 4. Local Outlier Factor (Novelty)
lof = LocalOutlierFactor(n_neighbors=15, contamination=0.06, novelty=True).fit(X_tr_normal)
lof_preds = np.where(lof.predict(X_te_s) == -1, 1, 0)
lof_scores = -lof.decision_function(X_te_s)

plt.figure(figsize=(8, 6))
rows = []
for name, preds, scores in [
    ('Extended Isolation Forest (EIF)', eif_preds, eif_scores),
    ('Standard Isolation Forest', std_preds, std_scores),
    ('One-Class SVM', svm_preds, svm_scores),
    ('Local Outlier Factor (LOF)', lof_preds, lof_scores),
]:
    auc = roc_auc_score(y_te, scores)
    fpr, tpr, _ = roc_curve(y_te, scores)
    plt.plot(fpr, tpr, lw=2, label=f'{name} (AUC = {auc:.4f})')
    rows.append({
        'Model 1 Algorithm': name,
        'Accuracy': accuracy_score(y_te, preds),
        'Precision': precision_score(y_te, preds),
        'Recall': recall_score(y_te, preds),
        'F1-Score': f1_score(y_te, preds),
        'ROC-AUC': auc,
    })

plt.plot([0, 1], [0, 1], 'k--', alpha=0.5)
plt.title('ROC Curve - Phát hiện Bất thường Tức thời (Model 1: EIF vs Baselines)', fontweight='bold')
plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR)')
plt.legend(loc='lower right')
plt.show()

res_df = pd.DataFrame(rows).sort_values('F1-Score', ascending=False)
display(res_df.round(4))

In [ ]:
print('🏆 Báo cáo Phân loại Chi tiết của Extended Isolation Forest (EIF):')
print(classification_report(y_te, eif_preds, target_names=['Ổn định (0)', 'Bất thường Tức thời (1)'], digits=4))

cm = confusion_matrix(y_te, eif_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', xticklabels=['Dự đoán Ổn định (0)', 'Dự đoán Bất thường (1)'], yticklabels=['Thực tế Ổn định (0)', 'Thực tế Bất thường (1)'])
plt.title('Confusion Matrix - Model 1 (Extended Isolation Forest)', fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Đóng gói & Xuất File `model1_eif_detector.pkl`

In [ ]:
final_scaler = RobustScaler().fit(X[y == 0])
X_all_s = np.arcsinh(final_scaler.transform(X))

final_eif = ExtendedIsolationForest(n_estimators=360, n_rotations=12, extension_level=len(MODEL1_FEATURES)-1, contamination=0.06, random_state=42)
final_eif.fit(X_all_s[y == 0])
all_scores = final_eif.anomaly_score(X_all_s)
best_th_all, best_f1_all = final_eif.threshold_, -1.0
for q in np.linspace(30, 65, 71):
    th = float(np.percentile(all_scores, q))
    f1_c = f1_score(y, (all_scores >= th).astype(int), zero_division=0)
    if f1_c > best_f1_all:
        best_f1_all, best_th_all = f1_c, th
final_eif.threshold_ = best_th_all

artifact_m1 = {
    'model_type': 'ExtendedIsolationForest',
    'model': final_eif,
    'scaler': final_scaler,
    'feature_names': MODEL1_FEATURES,
    'target_name': TARGET_COL,
}

joblib.dump(artifact_m1, 'model1_eif_detector.pkl')
print('✅ Đã xuất thành công Model 1: model1_eif_detector.pkl')